# Module 3 — Control Flow: Conditions, Loops and Comprehensions

**Python for Machine Learning · MSc Artificial Intelligence · Swansea University**

---

## Learning objectives

1. Make decisions with `if` / `elif` / `else` and conditional expressions.
2. Repeat work with `for` and `while` loops; control them with `break`, `continue` and `else`.
3. Use the built-ins `range`, `enumerate`, `zip`, `sorted`, `reversed`, `any`, `all`.
4. Write list, dictionary and set **comprehensions**.
5. Recognise the patterns behind ML code: training loops, early stopping, grid search.

---

## 3.1 Indentation defines blocks

Python uses **indentation** (4 spaces by convention) instead of braces `{}` to mark which lines belong to a block. A block is introduced by a line ending in a colon `:`.

```text
if condition:
    statement_inside      # indented -> part of the if-block
    another_inside
statement_after           # not indented -> always runs
```

Wrong indentation is a *syntax error*, or worse, a silent logic error. Jupyter and most editors indent automatically after `:`.

## 3.2 `if`, `elif`, `else`

Conditions are evaluated top to bottom; **only the first true branch runs**.

In [1]:
accuracy = 0.83

if accuracy >= 0.9:
    verdict = "excellent"
elif accuracy >= 0.8:
    verdict = "good"
elif accuracy >= 0.6:
    verdict = "fair"
else:
    verdict = "poor"

print(f"accuracy {accuracy:.0%} -> {verdict}")

accuracy 83% -> good


Conditions can combine `and`, `or`, `not`, and use truthiness:

In [2]:
data = [3, 5, 7]
model = None

if data and model is None:
    print("We have data but no model: time to train!")

if not data:
    print("Dataset is empty")   # does not run, because data is non-empty

We have data but no model: time to train!


### Conditional expression (ternary operator)

A compact one-line `if/else` that produces a **value**:

In [3]:
prob = 0.73
label = "positive" if prob >= 0.5 else "negative"
print(label)

positive


### `match` (structural pattern matching, Python 3.10+)

Useful when comparing one value against several options:

In [4]:
optimizer = "adam"

match optimizer:
    case "sgd":
        lr = 0.1
    case "adam" | "adamw":       # | means "or"
        lr = 0.001
    case _:                      # _ matches anything (default)
        lr = 0.01
print(lr)

0.001


## 3.3 `for` loops

A `for` loop iterates over **any iterable**: list, tuple, string, set, dictionary, range, file, …

In [5]:
models = ["kNN", "SVM", "Random Forest"]
for m in models:
    print("Training", m)

Training kNN
Training SVM
Training Random Forest


In [6]:
for ch in "ML":
    print(ch)

params = {"lr": 0.01, "epochs": 5}
for key in params:              # iterating over a dict gives its KEYS
    print(key, "->", params[key])

M
L
lr -> 0.01
epochs -> 5


### `range`

`range(stop)`, `range(start, stop)`, `range(start, stop, step)`. Like slicing, **stop is excluded**.

In [7]:
print(list(range(5)))
print(list(range(2, 8)))
print(list(range(0, 20, 5)))
print(list(range(10, 0, -3)))

[0, 1, 2, 3, 4]
[2, 3, 4, 5, 6, 7]
[0, 5, 10, 15]
[10, 7, 4, 1]


### `enumerate`: index **and** value

Avoid `for i in range(len(x))`. Prefer `enumerate`:

In [8]:
losses = [0.9, 0.6, 0.4]
for epoch, loss in enumerate(losses, start=1):
    print(f"epoch {epoch}: loss {loss}")

epoch 1: loss 0.9
epoch 2: loss 0.6
epoch 3: loss 0.4


### `zip`: iterate over several sequences in parallel

In [9]:
y_true = [1, 0, 1, 1, 0]
y_pred = [1, 0, 0, 1, 1]

correct = 0
for t, p in zip(y_true, y_pred):
    if t == p:
        correct += 1
print("accuracy:", correct / len(y_true))

accuracy: 0.6


`zip` stops at the shortest sequence. It is also handy to build dictionaries: `dict(zip(keys, values))`.

### Nested loops — e.g. a grid search

In [10]:
learning_rates = [0.1, 0.01]
depths = [3, 5, 7]

for lr in learning_rates:
    for d in depths:
        print(f"try lr={lr}, depth={d}")

try lr=0.1, depth=3
try lr=0.1, depth=5
try lr=0.1, depth=7
try lr=0.01, depth=3
try lr=0.01, depth=5
try lr=0.01, depth=7


The number of combinations is the product of the list lengths (here 2 × 3 = 6). In ML this explodes quickly — that is why random search exists.

### A simple "training loop"

The skeleton below is the pattern behind almost every ML training procedure: repeat an update, track the loss, report progress. Here we minimise $f(w) = (w - 3)^2$ with **gradient descent**: $w \leftarrow w - \eta f'(w)$ where $f'(w) = 2(w-3)$.

In [11]:
w = 0.0          # initial parameter
eta = 0.1        # learning rate
history = []

for step in range(25):
    loss = (w - 3) ** 2
    grad = 2 * (w - 3)
    w = w - eta * grad
    history.append(loss)
    if step % 5 == 0:
        print(f"step {step:2d}  w={w:.4f}  loss={loss:.6f}")

print("final w:", round(w, 4))

step  0  w=0.6000  loss=9.000000
step  5  w=2.2136  loss=0.966368
step 10  w=2.7423  loss=0.103763
step 15  w=2.9156  loss=0.011141
step 20  w=2.9723  loss=0.001196
final w: 2.9887


## 3.4 `while` loops

A `while` loop repeats **as long as** a condition is true. Use it when you do not know in advance how many iterations you need. Make sure the condition eventually becomes false, or the loop runs forever (in Jupyter: *Kernel → Interrupt*).

In [12]:
w, eta, step = 0.0, 0.1, 0
tolerance = 1e-6

while (w - 3) ** 2 > tolerance:
    w -= eta * 2 * (w - 3)
    step += 1

print(f"converged after {step} steps, w = {w:.5f}")

converged after 36 steps, w = 2.99903


## 3.5 `break`, `continue` and the loop `else`

* `break` — exit the loop immediately.
* `continue` — skip the rest of this iteration and go to the next.
* `else` on a loop — runs only if the loop finished **without** `break`.

In [13]:
values = [4, -1, 7, 0, 9]
for v in values:
    if v < 0:
        continue          # skip negative values
    if v == 0:
        print("found zero, stopping")
        break
    print("processing", v)

processing 4
processing 7
found zero, stopping


### Early stopping

A classic ML technique: stop training when the validation loss has not improved for `patience` epochs.

In [14]:
val_losses = [0.90, 0.75, 0.66, 0.61, 0.62, 0.60, 0.63, 0.64, 0.65, 0.59]
patience = 3
best = float("inf")      # positive infinity: any loss will be better
wait = 0

for epoch, loss in enumerate(val_losses):
    if loss < best:
        best, wait = loss, 0
        print(f"epoch {epoch}: improved to {loss}")
    else:
        wait += 1
        if wait >= patience:
            print(f"epoch {epoch}: no improvement for {patience} epochs -> early stop")
            break
else:
    print("trained for all epochs")

print("best validation loss:", best)

epoch 0: improved to 0.9
epoch 1: improved to 0.75
epoch 2: improved to 0.66
epoch 3: improved to 0.61
epoch 5: improved to 0.6
epoch 8: no improvement for 3 epochs -> early stop
best validation loss: 0.6


## 3.6 Helpful built-ins for iteration

In [15]:
nums = [3, 8, -2, 5]
print(sorted(nums), list(reversed(nums)))
print(any(n < 0 for n in nums))    # is at least one negative?
print(all(n < 10 for n in nums))   # are all below 10?
print(max(nums), min(nums), sum(nums))

words = ["banana", "kiwi", "apple"]
print(sorted(words, key=len))      # sort by length
print(max(words, key=len))

[-2, 3, 5, 8] [5, -2, 8, 3]
True
True
8 -2 14
['kiwi', 'apple', 'banana']
banana


## 3.7 Comprehensions

A comprehension builds a new collection from an iterable in one readable line. It is the **Pythonic** replacement for the "create empty list → loop → append" pattern.

In [16]:
# Loop version
squares = []
for x in range(6):
    squares.append(x ** 2)
print(squares)

# Comprehension version: [expression for item in iterable]
squares = [x ** 2 for x in range(6)]
print(squares)

[0, 1, 4, 9, 16, 25]
[0, 1, 4, 9, 16, 25]


### With a filter

`[expression for item in iterable if condition]`

In [17]:
scores = [55, 78, 91, 42, 67, 88]
passed = [s for s in scores if s >= 60]
print(passed)

[78, 91, 67, 88]


### With a conditional expression (transform every item)

Note the different position: the `if ... else` goes **before** `for`.

In [18]:
labels = ["pass" if s >= 60 else "fail" for s in scores]
print(labels)

['fail', 'pass', 'pass', 'fail', 'pass', 'pass']


### Dictionary and set comprehensions

In [19]:
names = ["ana", "ben", "chen"]
lengths = {n: len(n) for n in names}
print(lengths)

word_list = ["Data", "data", "DATA", "model"]
unique_lower = {w.lower() for w in word_list}
print(unique_lower)

# Inverting a dictionary: label -> index becomes index -> label
label_to_idx = {"cat": 0, "dog": 1, "bird": 2}
idx_to_label = {i: lab for lab, i in label_to_idx.items()}
print(idx_to_label)

{'ana': 3, 'ben': 3, 'chen': 4}
{'data', 'model'}
{0: 'cat', 1: 'dog', 2: 'bird'}


### Nested comprehensions

In [20]:
matrix = [[1, 2, 3], [4, 5, 6]]
flat = [x for row in matrix for x in row]          # read left-to-right like nested loops
transposed = [[row[j] for row in matrix] for j in range(3)]
print(flat)
print(transposed)

[1, 2, 3, 4, 5, 6]
[[1, 4], [2, 5], [3, 6]]


### Generator expressions

With **round brackets**, you get a generator: values are produced one at a time instead of building a whole list in memory. Great for large data passed straight to `sum`, `max`, `any`…

In [21]:
total = sum(x ** 2 for x in range(1_000_000))
print(total)

333332833333500000


> **Readability rule:** if a comprehension does not fit comfortably on one or two lines, write a normal loop.

---

## Summary

* Indentation defines blocks. `if/elif/else` runs the first true branch.
* `for` iterates over iterables; use `range`, `enumerate` and `zip` instead of manual index handling.
* `while` repeats until a condition becomes false; `break`/`continue` control the flow.
* Comprehensions build lists/dicts/sets concisely: `[f(x) for x in xs if cond(x)]`.

---

## Exercises

Solutions: `solutions/03_control_flow_solutions.ipynb`.

**Exercise 3.1 — Grade classifier.**
Swansea MSc marks: 70+ = Distinction, 60–69 = Merit, 50–59 = Pass, below 50 = Fail. For each mark in `marks`, print the mark and its classification.

In [22]:
marks = [72, 58, 49, 65, 70, 50, 81]
# Your code here

**Exercise 3.2 — FizzBuzz.**
For numbers 1 to 30: print `"Fizz"` if divisible by 3, `"Buzz"` if divisible by 5, `"FizzBuzz"` if divisible by both, otherwise the number itself.

In [23]:
# Your code here

**Exercise 3.3 — Confusion matrix counts.**
For binary labels (1 = positive), count true positives (TP), true negatives (TN), false positives (FP) and false negatives (FN) using `zip`. Then compute **accuracy** $= (TP+TN)/N$, **precision** $= TP/(TP+FP)$ and **recall** $= TP/(TP+FN)$.

In [24]:
y_true = [1, 0, 1, 1, 0, 1, 0, 0, 1, 1]
y_pred = [1, 0, 0, 1, 0, 1, 1, 0, 1, 0]
# Your code here

**Exercise 3.4 — Comprehensions.**
Using comprehensions only:
(a) the cubes of the even numbers from 0 to 20;
(b) from `words`, a list of the words longer than 4 characters, in upper case;
(c) a dictionary mapping each word in `words` to its number of vowels;
(d) the set of first letters of all words.

In [25]:
words = ["neural", "net", "tensor", "loss", "gradient", "epoch", "bias"]
# Your code here

**Exercise 3.5 — Gradient descent with a `while` loop.**
Minimise $f(x) = x^2 - 4x + 7$ (derivative $f'(x) = 2x - 4$), starting from `x = 10` with learning rate `0.05`. Stop when $|f'(x)| < 10^{-4}$ **or** after 1000 iterations. Print the number of iterations, the final `x` and $f(x)$. (The true minimum is at $x = 2$, $f = 3$.)

In [26]:
# Your code here

**Exercise 3.6 — Grid search.**
For the "validation score" function below, try every combination of `a` in `[0.1, 0.5, 1.0, 2.0]` and `b` in `[1, 2, 3, 4]`, and report the best combination and its score.

In [27]:
def val_score(a, b):
    return -(a - 0.5) ** 2 - (b - 3) ** 2 + 1   # a pretend validation score (higher is better)

# Your code here

**Exercise 3.7 — Prime numbers (challenge).**
Print all prime numbers below 60 using nested loops and the loop `else` clause. Then produce the same list with a single list comprehension using `all(...)`.

In [28]:
# Your code here